In [2]:
# Install datasets if needed and load both datasets
from datasets import load_dataset
import pandas as pd

# Load HaluEval - QA subset
print("Loading HaluEval...")
halueval = load_dataset("pminervini/HaluEval", "qa_samples", split="data")
print(f"HaluEval loaded: {len(halueval)} samples")
print("Columns:", halueval.column_names)
print()

# Preview
df_halu = pd.DataFrame(halueval)
print(df_halu.head(3))

d:\Nlp_Assignment\Nlp_Assignment\new_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading HaluEval...


d:\Nlp_Assignment\Nlp_Assignment\new_env\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ANP\.cache\huggingface\hub\datasets--pminervini--HaluEval. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating data split: 100%|██████████| 10000/10000 [00:00<00:00, 376586.19 examples/s]


HaluEval loaded: 10000 samples
Columns: ['knowledge', 'question', 'answer', 'hallucination']

                                           knowledge  \
0  Arthur's Magazine (1844–1846) was an American ...   
1  The Oberoi family is an Indian family that is ...   
2  Allison Beth "Allie" Goertz (born March 2, 199...   

                                            question  \
0  Which magazine was started first Arthur's Maga...   
1  The Oberoi family is part of a hotel company t...   
2  Musician and satirist Allie Goertz wrote a son...   

                               answer hallucination  
0  First for Women was started first.           yes  
1                               Delhi            no  
2             President Richard Nixon            no  


In [ ]:
# Check hallucination distribution
print("Hallucination label distribution:")
print(df_halu['hallucination'].value_counts())
print()

# Now load RAGTruth
print("Loading RAGTruth...")
ragtruth = load_dataset("wandb/RAGTruth-processed", split="train")
print(f"RAGTruth loaded: {len(ragtruth)} samples")
print("Columns:", ragtruth.column_names)
print()

df_rag = pd.DataFrame(ragtruth)
print(df_rag.head(3))

In [ ]:
from datasets import load_dataset

# Try correct RAGTruth path
ragtruth = load_dataset("wandb/ragtruth-data", split="train")
print(f"RAGTruth loaded: {len(ragtruth)} samples")
print("Columns:", ragtruth.column_names)

df_rag = pd.DataFrame(ragtruth)
print(df_rag.head(3))

In [ ]:
ragtruth = load_dataset("iamketan25/rag-hallucination-dataset", split="train")
print(f"Loaded: {len(ragtruth)} samples")
print("Columns:", ragtruth.column_names)
df_rag = pd.DataFrame(ragtruth)
print(df_rag.head(3))

In [ ]:
ragtruth = load_dataset("wandb/RAGTruth-processed", split="train")
print(f"RAGTruth loaded: {len(ragtruth)} samples")
print("Columns:", ragtruth.column_names)
df_rag = pd.DataFrame(ragtruth)
print(df_rag.head(3))
print("\nAll columns:", df_rag.columns.tolist())|

In [ ]:
ragtruth = load_dataset("wandb/RAGTruth-processed", split="train")
print(f"RAGTruth loaded: {len(ragtruth)} samples")
print("Columns:", ragtruth.column_names)
df_rag = pd.DataFrame(ragtruth)
print(df_rag.head(3))
print("\nAll columns:", df_rag.columns.tolist())


In [ ]:
import ast

# Check hallucination label structure
print("Sample hallucination_labels:")
print(df_rag['hallucination_labels'].iloc[0])
print()
print("Sample hallucination_labels_processed:")
print(df_rag['hallucination_labels_processed'].iloc[0])
print()

# Check task types (needed for Experiment 5)
print("Task types:", df_rag['task_type'].unique())
print()

# Check models (needed for Experiment 6)
print("Models:", df_rag['model'].unique())
print()

# Check quality distribution
print("Quality distribution:")
print(df_rag['quality'].value_counts())

In [ ]:
# Create binary hallucination label
df_rag['is_hallucinated'] = df_rag['hallucination_labels'].apply(lambda x: 1 if len(x) > 0 else 0)

print("RAGTruth hallucination distribution:")
print(df_rag['is_hallucinated'].value_counts())
print()

# Create hallucination type column for Experiment 5
def get_halu_type(row):
    if row['is_hallucinated'] == 0:
        return 'none'
    labels = row['hallucination_labels_processed']
    if labels.get('evident_conflict', 0) > 0:
        return 'contradictory'
    elif labels.get('baseless_info', 0) > 0:
        return 'unsupported'
    else:
        return 'fabricated'

df_rag['halu_type'] = df_rag.apply(get_halu_type, axis=1)
print("Hallucination type breakdown:")
print(df_rag['halu_type'].value_counts())
print()

# Save both datasets to disk
df_rag.to_csv('data/ragtruth.csv', index=False)
df_halu.to_csv('data/halueval.csv', index=False)
print("Both datasets saved to /data folder.")

In [ ]:
# Inspect the actual structure more carefully
print("Type of hallucination_labels[0]:", type(df_rag['hallucination_labels'].iloc[0]))
print("Value:", df_rag['hallucination_labels'].iloc[0])
print()

# Check a few more samples
for i in range(5):
    val = df_rag['hallucination_labels'].iloc[i]
    print(f"Sample {i}: type={type(val)}, value={val}, len={len(val)}")

In [ ]:
import json

def parse_labels(val):
    try:
        parsed = json.loads(val)
        return parsed
    except:
        return []

# Fix the labels
df_rag['labels_parsed'] = df_rag['hallucination_labels'].apply(parse_labels)
df_rag['is_hallucinated'] = df_rag['labels_parsed'].apply(lambda x: 1 if len(x) > 0 else 0)

print("RAGTruth hallucination distribution (fixed):")
print(df_rag['is_hallucinated'].value_counts())
print()

# Fix hallucination type
def get_halu_type(row):
    if row['is_hallucinated'] == 0:
        return 'none'
    for label in row['labels_parsed']:
        lt = label.get('label_type', '').lower()
        if 'conflict' in lt:
            return 'contradictory'
        elif 'baseless' in lt:
            return 'unsupported'
    return 'fabricated'

df_rag['halu_type'] = df_rag.apply(get_halu_type, axis=1)
print("Hallucination type breakdown (fixed):")
print(df_rag['halu_type'].value_counts())

# Re-save with fixed labels
df_rag.to_csv('data/ragtruth.csv', index=False)
print("\nDataset re-saved with correct labels.")